# Scope

- Add Human in Loop
- Improved instrumentations

In [ ]:
!pip install langgraph > /dev/null
!pip install langchain_openai > /dev/null
!pip install langchain_google_genai > /dev/null

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, Send, interrupt
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, ToolMessage, SystemMessage, BaseMessage
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint import memory
from langchain_core.tools import tool
from __future__ import annotations
import html
import ast
import difflib
import contextlib
import io
import math
import random
import re
import time
import json
import operator
import re
import yaml
import uuid
import traceback
import hashlib
from textwrap import dedent
from dataclasses import dataclass, field
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
from pydantic import BaseModel, Field
from typing import TypedDict, Annotated, List, Dict, Union, Any, Tuple, Callable

In [ ]:
import os
from google.colab import userdata
from pathlib import Path
from pathlib import Path
from typing import Optional, Literal
from langchain.tools import tool
from langchain.tools import InjectedToolArg
from dataclasses import dataclass
from langgraph.prebuilt import ToolNode, tools_condition
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

In [ ]:
from IPython.display import Markdown, display, Image

In [ ]:
import yaml

In [ ]:
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.text import Text
from rich.tree import Tree
from rich import box

In [ ]:
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['GOOGLE_API_KEY'] = userdata.get('API_KEY')

## Utilities

In [ ]:
def file_reducer(left, right):
    if not left:
        return right
    if not right:
        return left
    else:
        return {**left, **right}
def approx_tokens(text: str) -> int:
    # Rough heuristic: ~1 token per 4 chars
    return max(1, len(text) // 4)

def msg_tokens(m: BaseMessage) -> int:
    return approx_tokens(getattr(m, "content", "") or "")

In [ ]:
def compact_tool_message(m: ToolMessage, max_chars: int) -> ToolMessage:
    txt = m.content or ""
    if len(txt) <= max_chars:
        return m
    snippet = txt[:max_chars] + "\n...[truncated]..."
    return ToolMessage(content=snippet, tool_call_id=m.tool_call_id)

In [ ]:
def _messages_to_compact_text(msgs: List[BaseMessage], max_chars: int = 12000) -> str:
    parts = []
    for m in msgs:
        role = m.__class__.__name__.replace("Message", "").lower()
        content = getattr(m, "content", "") or ""
        parts.append(f"{role}: {content}")
    text = "\n".join(parts)
    if len(text) > max_chars:
        text = text[:max_chars] + "\n...[truncated]..."
    return text

In [ ]:
def _pretty_json(obj: Any) -> str:
    return json.dumps(obj, indent=2, ensure_ascii=False, default=str)

def _msg_role(m) -> str:
    return m.__class__.__name__.replace("Message", "").lower()

def _msg_preview(m, max_chars=600) -> str:
    c = getattr(m, "content", "") or ""
    if len(c) > max_chars:
        c = c[:max_chars] + "\n...[truncated]..."
    return c

In [ ]:
def _now_ms() -> int:
    return int(time.time() * 1000)

def _safe_len(x: Any) -> int:
    try:
        return len(x)  # type: ignore[arg-type]
    except Exception:
        return 0

def _coarse_size(obj: Any, max_items: int = 64) -> Dict[str, Any]:
    """
    Safe, cheap size estimator (no deep recursion).
    Returns lengths + approximate char sizes where possible.
    """
    try:
        if isinstance(obj, str):
            return {"type": "str", "chars": len(obj)}
        if isinstance(obj, list):
            return {"type": "list", "len": len(obj)}
        if isinstance(obj, dict):
            keys = list(obj.keys())[:max_items]
            return {"type": "dict", "len": len(obj), "keys_head": keys}
        return {"type": type(obj).__name__}
    except Exception:
        return {"type": "unknown"}

def _fingerprint_prompt(messages: List[Any], max_msgs: int = 12, max_chars: int = 8000) -> str:
    """
    Short fingerprint to detect prompt drift without storing whole prompt.
    """
    parts: List[str] = []
    for m in messages[-max_msgs:]:
        role = m.__class__.__name__
        txt = normalize_content(getattr(m, "content", None))
        parts.append(f"{role}:{txt[:800]}")
    blob = "\n".join(parts)[:max_chars].encode("utf-8")
    return hashlib.sha1(blob).hexdigest()

def _classify_error(exc: BaseException) -> str:
    """
    Coarse failure taxonomy useful for dashboards + retry policies.
    """
    name = type(exc).__name__.lower()
    msg = str(exc).lower()
    if "validationerror" in name or "pydantic" in msg:
        return "schema_validation"
    if "timeout" in name or "timed out" in msg:
        return "timeout"
    if "ratelimit" in msg or "429" in msg:
        return "rate_limit"
    if "permission" in msg or "denied" in msg:
        return "permission"
    if "filenotfound" in name or "no such file" in msg:
        return "file_not_found"
    if "connection" in name or "dns" in msg or "ssl" in msg:
        return "network"
    return "unknown"

## AgentState & Data Models

In [ ]:
@dataclass
class PythonReplResult:
    """Structured result returned by the python_repl tool."""
    stdout: str
    result_repr: str
    error: str
    elapsed_ms: int

In [ ]:
@dataclass
class PromptCard:
    name: str
    text: str
    tags: set[str] = field(default_factory=set)
    priority: int = 50  # lower = earlier

@dataclass
class PromptLibrary:
    cards: List[PromptCard]

    def select(self, selector: Callable[[PromptCard], bool]) -> List[PromptCard]:
        return sorted([c for c in self.cards if selector(c)], key=lambda c: c.priority)


In [ ]:
@dataclass
class SkillMeta:
    name: str
    description: str
    path: Path
    meta: Dict[str, Any]
    body: Optional[str] = None  # loaded on demand only


In [ ]:
@dataclass
class StepSnapshot:
    step_idx: int
    node: str
    update: Dict[str, Any]                     # per-node diff/update (stream_mode="updates")
    state_view: Dict[str, Any] = field(default_factory=dict)  # selected state keys

@dataclass
class Step:
    idx: int
    state: Dict[str, Any]          # full snapshot (values stream)
    diff: Dict[str, Any] = field(default_factory=dict)


In [ ]:
class VerifyRequest(BaseModel):
    reason: Literal["plan_created", "plan_changed", "clarification", "risky_action"] = "plan_changed"
    question: str = Field(..., description="What to ask the user.")
    kind: Literal["confirm", "clarify", "pick_one", "pick_many"] = "confirm"
    choices: Optional[List[str]] = None
    context: Optional[str] = Field(None, description="Plan snippet / diff / summary to show.")
    default: Optional[str] = Field(None, description="Default answer if UI supports quick submit.")

In [ ]:
class AgentState(TypedDict, total=False):
    history: List[AnyMessage]        # ground truth conversation
    messages: List[AnyMessage]   # curated messages passed to LLM
    memory: Dict[str, Any]               # summaries / episodic notes
    runtime: Dict[str, Any]              # turn flags, last tool, errors
    skills: List[SkillMeta]                 # rendered prompt chunk
    file_system: Annotated[Dict[str, Any], file_reducer]

## Policies

In [ ]:
@dataclass
class BudgetPolicy:
    max_prompt_tokens: int = 16000
    reserved_for_generation: int = 2000
    max_tool_snippet_chars: int = 1200
    max_skills_chars: int = 2500

@dataclass
class ToolLogPolicy:
    artifacts_dir: Path = Path("artifacts/tool_logs")
    max_inline_chars: int = 1200

@dataclass
class SummaryPolicy:
    summarize_when_history_len_exceeds: int = 50
    keep_last_n_messages: int = 18

@dataclass
class AppPolicy:
    artifacts_dir: Path = Path("artifacts/")

@dataclass
class ContextSignals:
    is_first_turn: bool
    after_tool: bool
    has_error: bool
    needs_planning: bool
    user_asked_capabilities: bool


## Skills

In [ ]:
_FRONTMATTER_RE = re.compile(r"^---\s*\n(.*?)\n---\s*\n(.*)$", re.DOTALL)

In [ ]:
def parse_skill_md(text: str, path: Path) -> SkillMeta:
    m = _FRONTMATTER_RE.match(text)
    if not m:
        raise ValueError(f"{path}: Missing YAML frontmatter (--- ... ---)")
    fm_raw, body = m.group(1), m.group(2)
    meta = yaml.safe_load(fm_raw) or {}
    name = (meta.get("name") or "").strip()
    desc = (meta.get("description") or "").strip()
    if not name or not desc:
        raise ValueError(f"{path}: 'name' and 'description' are required in frontmatter")
    return SkillMeta(name=name, description=desc, path=path, meta=meta, body=body.strip())

def discover_skills(skills_root: Path = Path(".skills")) -> List[SkillMeta]:
    skills: List[SkillMeta] = []
    if not skills_root.exists():
        return skills

    for d in sorted(skills_root.iterdir()):
        if not d.is_dir():
            continue
        skill_file = d / "SKILL.md"
        if not skill_file.exists():
            continue
        text = skill_file.read_text(encoding="utf-8")
        sk = parse_skill_md(text, skill_file)
        sk.body = None  # registry only
        skills.append(sk)
    return skills


# =========================
# Skills registry rendering (Top-K)
# =========================

def score_skill(skill: SkillMeta, query: str) -> int:
    q = (query or "").lower()
    s = (skill.name + " " + skill.description).lower()
    score = 0
    for term in set(re.findall(r"[a-zA-Z0-9_]+", q)):
        if len(term) >= 3 and term in s:
            score += 2
    for tok in skill.name.lower().split():
        if tok in q:
            score += 3
    return score

def render_skills_topk(skills: List[SkillMeta], user_text: str, max_chars: int, k: int = 12) -> str:
    if not skills:
        return "Available skills: (none found)"
    ranked = sorted(skills, key=lambda sk: score_skill(sk, user_text), reverse=True)
    top = ranked[:k]
    lines = ["Available skills (load only when needed):"]
    for s in top:
        lines.append(f"- {s.name}: {s.description}")
    out = "\n".join(lines)
    if len(out) > max_chars:
        out = out[:max_chars] + "\n...[truncated]..."
    return out


## Tools

### File System Tools

In [ ]:
@tool
def read_file(path: str) -> str:
    """
    Read the full contents of a file.

    Use this tool when you need to inspect or understand an entire file.
    If the file does not exist, an error will be raised.

    Args:
        path: Absolute or relative path to the file.

    Returns:
        The full file contents as a string.
    """
    p = Path(path)

    if not p.exists():
        raise FileNotFoundError(f"File not found: {path}")

    return p.read_text()


@tool
def read_file_range(path: str, start_line: int, end_line: Optional[int] = None) -> str:
    """
    Read a specific range of lines from a file.

    Use this tool when the file is large or when you only need to inspect
    a portion of the file. Line numbers are 1-indexed.

    If end_line is not provided, the file will be read from start_line to the end.

    Args:
        path: Path to the file.
        start_line: Starting line number (1-indexed).
        end_line: Optional ending line number (inclusive).

    Returns:
        The requested file content as a string.
    """
    p = Path(path)

    if not p.exists():
        raise FileNotFoundError(f"File not found: {path}")

    lines = p.read_text().splitlines()

    start_idx = max(start_line - 1, 0)
    end_idx = end_line if end_line is not None else len(lines)

    return "\n".join(lines[start_idx:end_idx])


@tool
def write_file(path: str, content: str, mode: str = "overwrite") -> str:
    """
    Write content to a file.

    Use this tool to create new files or update existing ones.

    Modes:
    - overwrite: Replace the entire file contents.
    - append: Add content to the end of the file.

    Args:
        path: Path to the file.
        content: Content to write.
        mode: Write mode ("overwrite" or "append").

    Returns:
        Confirmation message describing the operation.
    """
    p = Path(path)

    if mode == "overwrite":
        p.write_text(content)
        return f"File written (overwrite): {path}"

    elif mode == "append":
        with p.open("a") as f:
            f.write(content)
        return f"Content appended to file: {path}"

    else:
        raise ValueError("Mode must be 'overwrite' or 'append'")

### REPL tools

In [ ]:
class _SafeImportError(ImportError):
    pass


def _safe_import(name: str, globals=None, locals=None, fromlist=(), level=0):
    """
    Restricted __import__ implementation.

    You can extend the allowlist as needed.
    """
    allow = {"math", "random", "re", "time"}
    root = name.split(".", 1)[0]
    if root not in allow:
        raise _SafeImportError(f"Import '{name}' is not allowed. Allowed: {sorted(allow)}")
    return __import__(name, globals, locals, fromlist, level)


def _split_last_expr(code: str) -> Tuple[str, Optional[str]]:
    """
    If the code ends with an expression, separate it so we can eval() it
    and return its value (REPL-like behavior).
    """
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return code, None

    if not tree.body:
        return code, None

    last = tree.body[-1]
    if isinstance(last, ast.Expr):
        # Remove last Expr statement from the exec part.
        exec_body = tree.body[:-1]
        exec_mod = ast.Module(body=exec_body, type_ignores=[])
        exec_code = ast.unparse(exec_mod) if exec_body else ""
        last_expr_code = ast.unparse(last.value)
        return exec_code, last_expr_code

    return code, None


def _make_safe_builtins() -> Dict[str, Any]:
    """
    Provide a minimal, safer builtins set.
    Remove file/network/process capabilities.
    """
    return {
        # basics
        "abs": abs,
        "all": all,
        "any": any,
        "bool": bool,
        "dict": dict,
        "enumerate": enumerate,
        "float": float,
        "int": int,
        "len": len,
        "list": list,
        "max": max,
        "min": min,
        "pow": pow,
        "print": print,
        "range": range,
        "repr": repr,
        "round": round,
        "set": set,
        "sorted": sorted,
        "str": str,
        "sum": sum,
        "tuple": tuple,
        "zip": zip,
        # allow limited importing
        "__import__": _safe_import,
    }


# @tool
# def python_repl(
#     code: str,
#     state: Annotated[Dict[str, Any], InjectedToolArg()],
#     timeout_s: float = 2.0,
#     max_output_chars: int = 8000,
# ) -> Dict[str, Any]:
@tool
def python_repl(code: str,
                state: Optional[Dict[str, Any]] = None,
                timeout_s: float = 2.0,
                max_output_chars: int = 8000) -> dict:
    """
    Execute Python code in a restricted REPL-like environment and return output.

    This tool is designed for agentic workflows where the model needs to run small
    calculations, manipulate simple data, or prototype logic.

    Security model:
    - Runs with a minimal builtins set (no open(), exec(), eval(), os, subprocess, etc.)
    - Imports are restricted to an allowlist: math, random, re, time
    - No filesystem, network, or shell access via Python

    REPL behavior:
    - Captures stdout from print().
    - If the final statement is a Python expression, it is evaluated and its value is returned.

    Persistence:
    - The `state` dict is injected by the runtime and used to persist variables across calls.
      (The LLM cannot directly provide/modify `state`; your runtime injects it.)

    Args:
        code: Python code to execute (may be multi-line).
        state: Injected persistent dict for variables across invocations.
        timeout_s: Soft timeout (best-effort). For hard timeouts, run tool in a separate process.
        max_output_chars: Truncate stdout to this many characters.

    Returns:
        A dict with keys:
          - stdout: captured printed output (possibly truncated)
          - result: string repr of last expression value (if any)
          - error: error string (empty if success)
          - elapsed_ms: execution time in milliseconds
    """
    start = time.time()
    state = state or {}

    # Prepare safe globals; locals are the persistent state.
    safe_globals = {
        "__builtins__": _make_safe_builtins(),
        # Convenient pre-imported modules (also allowed via import)
        "math": math,
        "random": random,
        "re": re,
        "time": time,
    }

    # Ensure state is a dict (runtime responsibility, but be defensive)
    if not isinstance(state, dict):
        raise TypeError("Injected `state` must be a dict.")

    stdout_buf = io.StringIO()
    result_repr = ""
    error = ""

    exec_part, last_expr = _split_last_expr(code)

    try:
        with contextlib.redirect_stdout(stdout_buf):
            # Best-effort timeout check (not a hard kill)
            # For truly safe timeouts, run this tool in a separate process with OS-level limits.
            if exec_part.strip():
                compiled = compile(exec_part, "<python_repl>", "exec")
                exec(compiled, safe_globals, state)

            if last_expr is not None and last_expr.strip():
                compiled_expr = compile(last_expr, "<python_repl>", "eval")
                val = eval(compiled_expr, safe_globals, state)
                result_repr = repr(val)

            # soft timeout check after running (best-effort)
            if (time.time() - start) > timeout_s:
                raise TimeoutError(f"Execution exceeded {timeout_s}s (soft timeout).")

    except Exception as e:
        error = f"{type(e).__name__}: {e}"

    stdout = stdout_buf.getvalue()
    if len(stdout) > max_output_chars:
        stdout = stdout[:max_output_chars] + "\n...[truncated]..."

    elapsed_ms = int(round((time.time() - start) * 1000))

    return {
        "stdout": stdout,
        "result": result_repr,
        "error": error,
        "elapsed_ms": elapsed_ms,
    }


repl_state = {}

# In LangChain / LangGraph executor call:
python_repl.invoke(
    {"code": "x = 10\nx * 3"})

{'stdout': '', 'result': '30', 'error': '', 'elapsed_ms': 0}

### System Tools

In [ ]:
@tool
def load_skill(skill_name: str) -> str:
    """
    Load a skill by name from .skills/<dir>/SKILL.md (or by matching frontmatter name)
    and return JSON: {name, description, body, meta}.
    """
    skills_root = Path(".skills")

    # 1) direct directory match
    candidate = skills_root / skill_name / "SKILL.md"
    if candidate.exists():
        text = candidate.read_text(encoding="utf-8")
        sk = parse_skill_md(text, candidate)
        return json.dumps({
            "name": sk.name,
            "description": sk.description,
            "body": sk.body,
            "meta": sk.meta,
        })

    # 2) scan and match by frontmatter name
    if skills_root.exists():
        for d in skills_root.iterdir():
            if not d.is_dir():
                continue
            f = d / "SKILL.md"
            if not f.exists():
                continue
            text = f.read_text(encoding="utf-8")
            sk = parse_skill_md(text, f)
            if sk.name.lower() == skill_name.lower():
                return json.dumps({
                    "name": sk.name,
                    "description": sk.description,
                    "body": sk.body,
                    "meta": sk.meta,
                })

    raise FileNotFoundError(f"Skill not found: {skill_name}")

In [ ]:
@tool
def verify_with_user(request: VerifyRequest) -> Dict[str, Any]:
    """
    Human-in-the-loop gate. When called, execution pauses until the user answers.
    Returns the user's answer to the agent.

    Use only when:
    - You have created a plan and want approval before executing it, OR
    - You changed the plan in a major way and need re-approval, OR
    - You need a key clarification to proceed, OR
    - You are about to perform a risky/irreversible action.

    IMPORTANT:
    - Provide a short context (plan snippet) to help user decide quickly.
    - Ask ONE precise question.
    """
    answer = interrupt({
        "type": request.kind,
        "reason": request.reason,
        "question": request.question,
        "choices": request.choices,
        "context": request.context,
        "default": request.default,
    })

    return {"approved": True, "answer": answer, "reason": request.reason}

## Context Manager

### Utilities

In [ ]:
def detect_signals(state: Dict[str, Any]) -> ContextSignals:
    hist: List[BaseMessage] = state.get("history", [])
    runtime = state.get("runtime", {}) or {}

    is_first = len(hist) == 0 or runtime.get("turn_index", 0) == 0
    after_tool = bool(runtime.get("after_tool", False))
    has_error = bool(runtime.get("last_error", ""))

    user_msg = next((m for m in reversed(hist) if isinstance(m, HumanMessage)), None)
    user_text = (user_msg.content if user_msg else "") or ""
    needs_planning = runtime.get("force_planning", False) or (len(user_text) > 280)

    user_asked_cap = any(k in user_text.lower() for k in [
        "what can you do", "capabilities", "skills", "tools available"
    ])

    return ContextSignals(
        is_first_turn=is_first,
        after_tool=after_tool,
        has_error=has_error,
        needs_planning=needs_planning,
        user_asked_capabilities=user_asked_cap,
    )

def memory_messages(memory: Dict[str, Any]) -> List[SystemMessage]:
    out: List[SystemMessage] = []
    if not memory:
        return out
    summary = memory.get("summary", "")
    if summary:
        out.append(SystemMessage(content="Conversation summary:\n" + summary))
    plan = memory.get("plan", "")
    if plan:
        out.append(SystemMessage(content="Current plan:\n" + plan))
    return out

def active_skill_messages(state: Dict[str, Any]) -> List[SystemMessage]:
    runtime = state.get("runtime", {}) or {}
    name = runtime.get("active_skill_name")
    body = runtime.get("active_skill_body")
    if not name or not body:
        return []
    return [SystemMessage(content=f"ACTIVE SKILL: {name}\n\n{body}")]


### Manager

In [ ]:
class ContextManager:
    def __init__(self, prompt_lib: PromptLibrary, budget: BudgetPolicy):
        self.prompt_lib = prompt_lib
        self.budget = budget

    def compose(self, state: Dict[str, Any]) -> List[BaseMessage]:
        hist: List[BaseMessage] = state.get("history", [])
        memory = state.get("memory", {}) or {}
        skills: List[SkillMeta] = state.get("skills", []) or []
        sig = detect_signals(state)

        # 1) Prompt cards (system stack)
        cards = self._select_cards(sig)
        system_msgs = [SystemMessage(content=c.text) for c in cards]

        # 2) Active skill injection (one skill body)
        active_msgs = active_skill_messages(state)

        # 3) Skills registry gating (Top-K)
        skills_msg: List[SystemMessage] = []
        if self._should_inject_skills(sig) and skills:
            user_msg = next((m for m in reversed(hist) if isinstance(m, HumanMessage)), None)
            user_text = (user_msg.content if user_msg else "") or ""
            skills_text = render_skills_topk(skills, user_text, self.budget.max_skills_chars, k=12)
            skills_msg = [SystemMessage(content=skills_text)]

        # 4) Memory
        mem_msgs = memory_messages(memory)

        # 5) History trimming + tool compaction (token budget will do the real trimming)
        curated_hist = self._curate_history(hist, sig)

        assembled = system_msgs + active_msgs + skills_msg + mem_msgs + curated_hist
        return self._fit_to_budget(assembled)

    def _select_cards(self, sig: ContextSignals) -> List[PromptCard]:
        tags = {"core"}
        if sig.after_tool:
            tags.add("after_tool")
        if sig.has_error:
            tags.add("error")
        if sig.needs_planning:
            tags.add("planning")
        return self.prompt_lib.select(lambda c: len(c.tags & tags) > 0)

    def _should_inject_skills(self, sig: ContextSignals) -> bool:
        return sig.is_first_turn or sig.user_asked_capabilities or sig.needs_planning or sig.has_error

    def _curate_history(self, hist: List[BaseMessage], sig: ContextSignals) -> List[BaseMessage]:
        if not hist:
            return []
        out: List[BaseMessage] = []
        for m in hist:
            if isinstance(m, ToolMessage):
                out.append(compact_tool_message(m, self.budget.max_tool_snippet_chars))
            else:
                out.append(m)
        return out

    def _fit_to_budget(self, msgs: List[BaseMessage]) -> List[BaseMessage]:
        max_in = max(1000, self.budget.max_prompt_tokens - self.budget.reserved_for_generation)

        system = [m for m in msgs if isinstance(m, SystemMessage)]
        others = [m for m in msgs if not isinstance(m, SystemMessage)]

        def total(ms: List[BaseMessage]) -> int:
            return sum(msg_tokens(m) for m in ms)

        kept_system = system
        if total(kept_system) > max_in:
            truncated: List[BaseMessage] = []
            running = 0
            for sm in kept_system:
                c = sm.content or ""
                allow_chars = max(200, (max_in - running) * 4)
                truncated.append(SystemMessage(
                    content=c[:allow_chars] + ("\n...[truncated]..." if len(c) > allow_chars else "")
                ))
                running = total(truncated)
                if running >= max_in:
                    return truncated
            kept_system = truncated

        kept_others: List[BaseMessage] = []
        running = total(kept_system)
        for m in reversed(others):
            t = msg_tokens(m)
            if running + t > max_in:
                continue
            kept_others.append(m)
            running += t
        kept_others.reverse()
        return kept_system + kept_others

## Nodes

In [ ]:
def persist_tool_outputs_node(state: Dict[str, Any], policy: ToolLogPolicy) -> Dict[str, Any]:
    history: List[BaseMessage] = state.get("history", [])
    runtime = state.get("runtime", {}) or {}
    run_id = runtime.get("run_id", "default_run")

    out_history: List[BaseMessage] = []
    changed = False

    base_dir = policy.artifacts_dir / run_id
    base_dir.mkdir(parents=True, exist_ok=True)

    for m in history:
        if isinstance(m, ToolMessage):
            content = m.content or ""
            if len(content) > policy.max_inline_chars:
                tool_call_id = m.tool_call_id or "unknown_tool_call"
                file_path = base_dir / f"{tool_call_id}.txt"
                file_path.write_text(content, encoding="utf-8")

                snippet = content[:policy.max_inline_chars] + "\n...[truncated]..."
                pointer = f"\n\nFull tool output saved at: {file_path.as_posix()}"
                out_history.append(ToolMessage(content=snippet + pointer, tool_call_id=m.tool_call_id))
                changed = True
            else:
                out_history.append(m)
        else:
            out_history.append(m)

    if not changed:
        return {}

    runtime = {**runtime, "after_tool": True, "tool_logs_persisted": True}
    return {"history": out_history, "runtime": runtime}


In [ ]:
def summarize_node(state: Dict[str, Any], *, llm, policy: SummaryPolicy) -> Dict[str, Any]:
    history: List[BaseMessage] = state.get("history", [])
    if len(history) <= policy.summarize_when_history_len_exceeds:
        return {}

    keep_n = policy.keep_last_n_messages
    old = history[:-keep_n]
    recent = history[-keep_n:]

    memory = state.get("memory", {}) or {}
    existing_summary = memory.get("summary", "")

    old_text = _messages_to_compact_text(old)

    prompt = [
        SystemMessage(content=(
            "You are a summarization engine for an agent runtime.\n"
            "Update the running summary so it preserves: user goals, constraints, decisions, tool outcomes, and open tasks.\n"
            "Be concise, structured, and factual. No fluff."
        )),
        HumanMessage(content=(
            f"Existing summary:\n{existing_summary}\n\n"
            f"New conversation chunk to merge:\n{old_text}\n\n"
            "Return an updated summary with headings:\n"
            "- Goals\n- Decisions\n- Tool outcomes\n- Open tasks\n- Constraints\n"
        )),
    ]

    resp = llm.invoke(prompt)
    updated_summary = resp.content

    memory = {**memory, "summary": updated_summary}
    runtime = {**(state.get("runtime", {}) or {}), "summarized": True}

    return {"memory": memory, "history": recent, "runtime": runtime}

In [ ]:
def activate_skill_from_tool_result_node(state: Dict[str, Any]) -> Dict[str, Any]:
    """
    If the most recent ToolMessage contains a load_skill JSON payload, set:
      runtime["active_skill_name"], runtime["active_skill_body"], runtime["active_skill_meta"]
    """
    hist = state.get("history", [])
    if not hist:
        return {}

    # Scan from end for the most recent load_skill tool output
    for m in reversed(hist):
        if not isinstance(m, ToolMessage):
            continue
        txt = m.content or ""
        try:
            payload = json.loads(txt)
        except Exception:
            continue

        if isinstance(payload, dict) and payload.get("body") and payload.get("name"):
            runtime = state.get("runtime", {}) or {}
            runtime = {
                **runtime,
                "active_skill_name": payload["name"],
                "active_skill_body": payload["body"],
                "active_skill_meta": payload.get("meta", {}),
            }
            return {"runtime": runtime}

    return {}

In [ ]:
def context_node(state: AgentState, ctx_mgr: ContextManager):
        messages = ctx_mgr.compose(state)
        return {"messages": messages}

In [ ]:
def llm_node(state: AgentState, llm: Optional[ChatGoogleGenerativeAI]):
        msgs = state.get("messages", [])
        ai = llm.invoke(msgs)
        hist = state.get("history", [])
        runtime = state.get("runtime", {}) or {}
        # After LLM call: turn_index increments, clear after_tool (tools will set it)
        runtime = {**runtime, "turn_index": runtime.get("turn_index", 0) + 1, "after_tool": False}
        return {"history": hist + [ai], "runtime": runtime}

In [ ]:
def instrument_node(
    name: str,
    fn: Callable[[Dict[str, Any]], Dict[str, Any]],
    *,
    # Which state keys you want to measure and optionally snapshot
    history_key: str = "history",
    prompt_key: str = "messages",   # your context node should set this
    # When True, also store a small snapshot of state deltas for debugging
    capture_state_sizes: bool = True,
    capture_tool_calls: bool = True,
    # Store only a fingerprint by default (safer). You can also store truncated text.
    capture_prompt_fingerprint: bool = True,
    capture_prompt_preview: bool = False,
    prompt_preview_chars: int = 1200,
    # Decide whether to swallow exceptions (and let graph continue) or re-raise
    swallow_exceptions: bool = False,
) -> Callable[[Dict[str, Any]], Dict[str, Any]]:
    """
    Wrap a LangGraph node function with production-grade observability.

    What it logs into state["telemetry"]:
      - run_id / trace_id / turn_index / step_id
      - node timing (start/end/elapsed_ms)
      - status ("ok" | "error") + error classification
      - updated keys returned by node
      - message counts + approximate token counts
      - tool calls observed in latest AI messages (optional)
      - prompt fingerprint (optional) to detect drift without storing prompt content

    Also updates runtime on error:
      runtime["last_error"], runtime["last_error_traceback"], runtime["last_failed_node"], runtime["after_tool"]=False

    Notes:
      - To make Prompt tab in your UI reliable, ensure your context node writes state[prompt_key].
      - If you persist prompts as artifacts, do that in a separate persist_prompt node.
    """

    def wrapped(state: Dict[str, Any]) -> Dict[str, Any]:
        # --- correlation ids ---
        runtime_in = state.get("runtime", {}) or {}
        trace_id = runtime_in.get("trace_id") or runtime_in.get("run_id") or str(uuid.uuid4())
        run_id = runtime_in.get("run_id") or trace_id
        turn = int(runtime_in.get("turn_index", 0))
        step_id = str(uuid.uuid4())

        t0 = time.perf_counter()
        start_ms = _now_ms()

        out: Dict[str, Any] = {}
        status = "ok"
        err_text = ""
        err_type = ""
        err_class = ""
        tb = ""

        try:
            out = fn(state) or {}
            return out
        except Exception as e:
            status = "error"
            err_type = type(e).__name__
            err_class = _classify_error(e)
            tb = traceback.format_exc()
            err_text = f"{err_type}: {e}"

            # record into runtime (so context manager can react)
            runtime = dict(runtime_in)
            runtime["last_error"] = err_text
            runtime["last_error_traceback"] = tb
            runtime["last_failed_node"] = name
            runtime["after_tool"] = False  # usually you're not "after tool" if node blew up

            # ensure runtime update persists even if node raises
            out = dict(out) if out else {}
            out["runtime"] = runtime

            if not swallow_exceptions:
                raise
            return out
        finally:
            end_ms = _now_ms()
            elapsed_ms = int((time.perf_counter() - t0) * 1000)

            # --- metrics from *input state* (stable) ---
            hist = state.get(history_key, []) or state.get("messages", []) or []
            pm = state.get(prompt_key, []) or []

            history_len = _safe_len(hist)
            prompt_len = _safe_len(pm)

            # token approximations
            history_tokens = 0
            prompt_tokens = 0
            try:
                history_tokens = sum(msg_tokens(m) for m in hist[-20:])  # recent window
            except Exception:
                history_tokens = 0
            try:
                prompt_tokens = sum(msg_tokens(m) for m in pm)
            except Exception:
                prompt_tokens = 0

            tool_calls: List[Dict[str, Any]] = []
            if capture_tool_calls:
                # scan last few AI messages for tool_calls
                try:
                    for m in reversed(hist[-10:]):
                        if m.__class__.__name__ == "AIMessage":
                            tool_calls = extract_tool_calls(m)
                            if tool_calls:
                                break
                except Exception:
                    tool_calls = []

            prompt_fingerprint = ""
            prompt_preview = ""
            if capture_prompt_fingerprint:
                try:
                    prompt_fingerprint = _fingerprint_prompt(pm)
                except Exception:
                    prompt_fingerprint = ""
            if capture_prompt_preview and pm:
                try:
                    # store only last system+human+ai-ish snippets; keep it short
                    tail = pm[-6:]
                    chunks = []
                    for m in tail:
                        role = m.__class__.__name__
                        txt = normalize_content(getattr(m, "content", None))
                        chunks.append(f"{role}:\n{txt[:300]}")
                    prompt_preview = "\n\n".join(chunks)[:prompt_preview_chars]
                except Exception:
                    prompt_preview = ""

            entry: Dict[str, Any] = {
                "trace_id": trace_id,
                "run_id": run_id,
                "turn_index": turn,
                "step_id": step_id,
                "node": name,
                "status": status,
                "start_ms": start_ms,
                "end_ms": end_ms,
                "elapsed_ms": elapsed_ms,
                "updates": sorted(list((out or {}).keys())),
                "counts": {
                    "history_len": history_len,
                    "prompt_len": prompt_len,
                },
                "tokens_approx": {
                    "history_recent_window": history_tokens,
                    "prompt_total": prompt_tokens,
                },
            }

            if capture_state_sizes:
                entry["state_sizes"] = {
                    history_key: _coarse_size(hist),
                    prompt_key: _coarse_size(pm),
                    "runtime": _coarse_size(runtime_in),
                }

            if capture_tool_calls:
                entry["tool_calls"] = tool_calls

            if capture_prompt_fingerprint:
                entry["prompt_fingerprint"] = prompt_fingerprint
            if capture_prompt_preview:
                entry["prompt_preview"] = prompt_preview

            if status == "error":
                entry["error"] = {
                    "type": err_type,
                    "class": err_class,
                    "message": err_text,
                    "traceback": tb,
                }

            telemetry = list(state.get("telemetry", []) or [])
            telemetry.append(entry)

            # Ensure telemetry persists
            if out is None:
                out = {}
            out["telemetry"] = telemetry

    return wrapped

In [ ]:
def tools_node(state: AgentState, tool_node_impl: ToolNode):
        """
        ToolNode expects {"messages": ...}. We'll feed it history and then append only the delta.
        """
        hist = state.get("history", [])
        res = tool_node_impl.invoke({"messages": hist})
        updated = res["messages"]
        # new_msgs = updated[len(hist):]
        runtime = state.get("runtime", {}) or {}
        runtime = {**runtime, "after_tool": True}
        return {"history": hist + updated, "runtime": runtime}

In [ ]:
def persist_prompt_artifact_node(state: AgentState) -> AgentState:
    runtime = state.get("runtime", {}) or {}
    turn = runtime.get("turn_index", 0)

    pm = get_prompt_messages_from_state(state)  # uses robust getter
    if not pm:
        return {}  # nothing to persist

    lines = []
    for m in pm:
        role = m.__class__.__name__.replace("Message", "").lower()
        lines.append(f"{role}:\n{normalize_content(getattr(m, 'content', None))}\n")

    prompt_text = "\n".join(lines)

    arts = list(runtime.get("prompt_artifacts", []) or [])
    arts.append({"turn": turn, "prompt_text": prompt_text})
    runtime["prompt_artifacts"] = arts

    return {"runtime": runtime}

### Routers

In [ ]:
def has_tool_calls(state: Dict[str, Any]) -> str:
    hist = state.get("history", [])
    if not hist:
        return "end"
    last = hist[-1]
    if isinstance(last, AIMessage) and getattr(last, "tool_calls", None):
        return "tools"
    return "end"

def should_summarize(state: Dict[str, Any], policy: SummaryPolicy) -> str:
    hist = state.get("history", [])
    return "summarize" if len(hist) > policy.summarize_when_history_len_exceeds else "skip"

In [ ]:
def should_pause(state: Dict[str, Any]) -> str:
    runtime = state.get("runtime", {}) or {}
    if runtime.get("waiting_for_user"):
        return "pause"
    return "continue"

## Graph

In [ ]:
def build_app(
    llm,  # e.g., ChatGoogleGenerativeAI(...).bind_tools(tools)
    prompt_lib: PromptLibrary,
    skills_root: Path = Path(".skills"),
    budget_policy: BudgetPolicy = BudgetPolicy(),
    tool_log_policy: ToolLogPolicy = ToolLogPolicy(),
    summary_policy: SummaryPolicy = SummaryPolicy(),
    tools: Optional[List[Any]] = None,
):
    """
    Build and compile the LangGraph app.

    Requirements:
      - llm must support .invoke(messages) and should be bound to tools:
          llm = llm.bind_tools(tools)
      - Tools list will include load_skill plus any extra tools you provide.

    State initialization you should do when invoking:
      {
        "history": [HumanMessage(...)],
        "memory": {},
        "runtime": {"run_id": "...", "turn_index": 0},
        "skills": discover_skills(Path(".skills")),
      }
    """
    ctx_mgr = ContextManager(prompt_lib=prompt_lib, budget=budget_policy)

    # Tools for ToolNode:
    tool_node_impl = ToolNode(tools)

    def summarize_wrapped(state: AgentState):
        return summarize_node(state, llm=llm, policy=summary_policy) or {}

    def context_node_wrapped(state: AgentState):
        return context_node(state, ctx_mgr) or {}

    def tools_node_wrapped(state: AgentState):
        return tools_node(state, tool_node_impl) or {}

    def persist_node(state: AgentState):
        return persist_tool_outputs_node(state, tool_log_policy) or {}

    def activate_skill_node(state: AgentState):
        return activate_skill_from_tool_result_node(state) or {}

    def llm_node_wrapped(state: AgentState):
        return llm_node(state, llm = llm) or {}

    # ---- Build graph ----
    graph = StateGraph(AgentState)

    graph.add_node("summarize", instrument_node("summarize", summarize_wrapped))
    graph.add_node("context", instrument_node("context", context_node_wrapped))
    graph.add_node("persist_prompt", instrument_node("persist_prompt", persist_prompt_artifact_node))
    graph.add_node("llm", instrument_node("llm", llm_node_wrapped))
    graph.add_node("tools", instrument_node("tools", tools_node_wrapped))
    graph.add_node("persist_tool_outputs", instrument_node("persist_tool_outputs", persist_node))
    graph.add_node("activate_skill", instrument_node("activate_skill", activate_skill_node))

    # Entry
    graph.set_entry_point("summarize")

    # summarize -> (summarize again or skip) -> context
    graph.add_conditional_edges(
        "summarize",
        lambda s: should_summarize(s, summary_policy),
        {"summarize": "summarize", "skip": "context"},
    )
    graph.add_edge("summarize", "context")
    graph.add_edge("context", "persist_prompt")
    graph.add_edge("persist_prompt", "llm")

    # llm -> tools or end
    graph.add_conditional_edges("llm", has_tool_calls, {"tools": "tools", "end": END})

    # tools -> persist -> activate_skill -> summarize -> context ...
    graph.add_edge("tools", "persist_tool_outputs")
    graph.add_edge("persist_tool_outputs", "activate_skill")
    graph.add_edge("activate_skill", "summarize")
    checkpointer = MemorySaver()
    return graph.compile(checkpointer = checkpointer)

In [ ]:
SYSTEM_PROMPT = dedent("""\
You are a helpful tool-using agent. Tool outputs are ground truth.
USERs can ask you question that may or may not require tool calling.
Use your judgement to identify best course of action.
""")
IDENTITY_PROMPT = dedent("""\
## Identity:
You are the root agent directly interfacing with the USER within agent runtime.
""")
GUIDELINES_PROMPT = dedent("""\
## Guidelines:
- Be direct. Use tools when needed. If blocked, ask one precise question.
- Always create a plan as a first step and save it as plan.md in artifacts directory
- IMPORTANT: Solve the problem only following the plan and don't bypass the plan
- After writing plan.md, ask the USER to approve it before executing any tool steps beyond planning.
- If blocked or uncertain about a key detail, ask one precise question and wait for the answer before proceeding.
- If you create plan.md for the first time, call verify_with_user(reason="plan_created") with a short plan summary and wait for approval before executing any non-planning tasks.
- If you later revise plan.md in a way that changes tasks (not just status), call verify_with_user(reason="plan_changed") and wait for approval.
- Do NOT call verify_with_user for status updates (pending/completed changes only).
- If blocked, call verify_with_user(reason="clarification") with one precise question.
- Major plan change = adding/removing tasks, changing task order significantly, or changing the chosen approach/tools.
- Minor change = only updating task status or adding small notes.
""")
AFTER_TOOLS_PROMPT = dedent("""\
## After a tool call:
After a tool call: integrate tool results, then decide next action.
If you are finished with the task, then respond with final answer.
""")
ERROR_RECOVERY_PROMPT = dedent("""\
## After an error:
If an error occurs: propose 2-3 fixes, choose safest default, retry if appropriate.
""")

PLANNING_PROMPT = dedent("""\
## Planning instructions:
Use these instructions to construct, refer to and update the plan:

### When to plan
- Create plan for tasks that require multi step task execution.
- Create a plan when users explicitly refer to executing multiple tasks.
- Create a plan when any tool call or skill reference is needed.
- DON'T CREATE PLAN WHICH REQUIRES NO TOOL CALL.
- Initially, list every task as pending.

### How to plan
- Create a plan to decompose problem into multi small problems.
- Each task in a plan can either be completely by a tool call or by referring to a skill which may requires multiple tool calls.

### How to update:
- Once a task is completed, revisit the plan to update the status of each task.
- Each task can only have a status of completed, in-progress or pending.
- For a completed task, update its status to completed.
-
### How to use the plan:
- Use suitable tools to read the plan and then identify next task.
- Execute only one task at a time from plan unless tasks can be run in parallel.
- After every task, refer to your plan and update task status.
- Identify if re-planning or plan update is needed.
- If no plan update is needed, then move onto the next task by changing its status from pending to in-progress.

### IMPORTANT: READ PLAN TO IDENTIFY THE NEXT TASK. EXECUTE ONLY ONE TASK AT A TIME. ALWAYS UPDATE THE STATUS OF THE TASK.

""")

In [ ]:
prompt_lib = PromptLibrary(cards=[
        PromptCard("system", SYSTEM_PROMPT, {"core"}, priority=0),
        PromptCard("identity", IDENTITY_PROMPT, {"core"}, priority=5),
        PromptCard("guidelines", GUIDELINES_PROMPT, {"core"}, priority=10),

        PromptCard("after_tool",  AFTER_TOOLS_PROMPT, {"after_tool"}, priority=20),
        PromptCard("error_recovery", ERROR_RECOVERY_PROMPT, {"error"}, priority=20),
        PromptCard("planning", PLANNING_PROMPT, {"core"}, priority=10),
    ])

In [ ]:
budget_policy = BudgetPolicy(max_prompt_tokens = 16000, reserved_for_generation = 2000,
                          max_tool_snippet_chars = 1200, max_skills_chars = 2500)
tool_policy = ToolLogPolicy(max_inline_chars = 1200)
summary_policy = SummaryPolicy(summarize_when_history_len_exceeds = 50, keep_last_n_messages = 18)
llm = ChatGoogleGenerativeAI(model = 'models/gemini-3-flash-preview')
ctx_mgr = ContextManager(prompt_lib, budget_policy)
tools = [load_skill] + [read_file, write_file, python_repl, verify_with_user]
llm_with_tools = llm.bind_tools(tools)
os.makedirs('artifacts', exist_ok=True)

In [ ]:
app = build_app(llm_with_tools, prompt_lib,
                budget_policy=budget_policy,
                tool_log_policy=tool_policy,
                summary_policy = summary_policy,
                tools = tools)

In [ ]:
state = {'history': [HumanMessage('Randomly decide on a ML application to develop a model for and then develop the model.')]}

In [ ]:
config = {
    "configurable": {
        "thread_id": "unique-session-id-123",
        "user_id": "user_456"
    }
}

In [ ]:
final_state = app.invoke(state, config = config)

In [ ]:
interrupts = final_state.get('__interrupt__')

In [ ]:
final_result = app.invoke(Command(resume = 'confirm'), config = config)

## Debug UI

### Utilities

In [ ]:
def _pretty_json(obj: Any) -> str:
    return json.dumps(obj, indent=2, ensure_ascii=False, default=str)

def _role(m) -> str:
    return m.__class__.__name__.replace("Message", "").lower()

def _content(m) -> str:
    return getattr(m, "content", "") or ""

def _truncate(s: str, n: int = 4000) -> str:
    return s if len(s) <= n else s[:n] + "\n...[truncated]..."

def approx_tokens(text: str) -> int:
    # rough heuristic: 1 token ~= 4 chars
    return max(1, len(text) // 4)

def msg_tokens(m) -> int:
    return approx_tokens(_content(m))

def extract_tool_calls(msg: Any) -> List[Dict[str, Any]]:
    """
    Extract tool calls from AI messages across providers.
    Checks:
      - msg.tool_calls (LC standard)
      - msg.additional_kwargs["tool_calls"]
      - msg.additional_kwargs["function_call"] (older OpenAI style)
    """
    calls = []

    tc = getattr(msg, "tool_calls", None)
    if tc:
        calls.extend(tc)

    ak = getattr(msg, "additional_kwargs", None) or {}
    if isinstance(ak, dict):
        if ak.get("tool_calls"):
            calls.extend(ak["tool_calls"])
        if ak.get("function_call"):
            calls.append({"name": ak["function_call"].get("name"), "args": ak["function_call"].get("arguments")})

    # Normalize to {name, args, id?}
    norm = []
    for c in calls:
        if isinstance(c, dict):
            norm.append({
                "id": c.get("id") or c.get("tool_call_id"),
                "name": c.get("name") or (c.get("function") or {}).get("name"),
                "args": c.get("args") or (c.get("function") or {}).get("arguments"),
            })
        else:
            norm.append({"name": str(c), "args": None})
    return norm

def safe_get(d: Dict[str, Any], k: str, default):
    v = d.get(k, default)
    return default if v is None else v

In [ ]:
def get_history_from_state(state: Dict[str, Any]) -> List[BaseMessage]:
    # supports graphs that store messages under different keys
    return state.get("history") or state.get("messages") or []

def get_prompt_messages_from_state(state: Dict[str, Any]) -> List[BaseMessage]:
    return state.get("messages") or state.get("input_messages") or state.get("llm_input") or []

def get_prompt_text_fallback(state: Dict[str, Any]) -> str:
    runtime = state.get("runtime", {}) or {}
    arts = runtime.get("prompt_artifacts") or []
    if arts:
        return arts[-1].get("prompt_text", "") or ""
    return ""

In [ ]:
def normalize_content(content: Any) -> str:
    """
    Provider-agnostic normalization of message.content into displayable text.
    Handles:
      - str
      - None
      - dict
      - list[blocks] (Gemini/OpenAI multimodal/tool blocks)
    """
    if content is None:
        return ""
    if isinstance(content, str):
        return content

    if isinstance(content, dict):
        # Sometimes content is a dict with text keys
        if "text" in content:
            return str(content["text"])
        return json.dumps(content, indent=2, ensure_ascii=False, default=str)

    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                # common keys
                if "text" in item:
                    parts.append(str(item["text"]))
                elif "content" in item:
                    parts.append(str(item["content"]))
                elif item.get("type") == "text" and "text" in item:
                    parts.append(str(item["text"]))
                else:
                    parts.append(json.dumps(item, ensure_ascii=False, default=str))
            else:
                parts.append(str(item))
        return "\n".join([p for p in parts if p is not None])

    return str(content)


In [ ]:
def _shallow_snapshot(state: Dict[str, Any], keys: List[str]) -> Dict[str, Any]:
    snap = {}
    for k in keys:
        if k in state:
            snap[k] = state[k]
    return snap

def _diff_states(prev: Dict[str, Any], cur: Dict[str, Any]) -> Dict[str, Any]:
    """
    Lightweight diff for a few keys.
    - For lists: show length change
    - For dicts: show changed keys
    - For scalars: show old/new
    """
    diff = {}
    all_keys = set(prev.keys()) | set(cur.keys())
    for k in sorted(all_keys):
        a = prev.get(k, None)
        b = cur.get(k, None)
        if a == b:
            continue
        if isinstance(a, list) and isinstance(b, list):
            diff[k] = {"type": "list", "prev_len": len(a), "cur_len": len(b)}
        elif isinstance(a, dict) and isinstance(b, dict):
            changed = sorted(set(a.keys()) ^ set(b.keys()) | {kk for kk in a.keys() & b.keys() if a[kk] != b[kk]})
            diff[k] = {"type": "dict", "changed_keys": changed[:100]}
        else:
            diff[k] = {"type": "value", "prev": str(a)[:200], "cur": str(b)[:200]}
    return diff

In [ ]:
def _pretty(obj):
    return json.dumps(obj, indent=2, ensure_ascii=False, default=str)

In [ ]:
def render_messages(msgs: List[Any], title: str, search: str = "", only_matches: bool = False) -> str:
    if not msgs:
        return f"_(empty: {title})_"

    q = (search or "").strip().lower()

    out = [f"## {title} ({len(msgs)} msgs)"]
    for i, m in enumerate(msgs):
        role = m.__class__.__name__.replace("Message", "").lower()
        raw = getattr(m, "content", None)
        text = normalize_content(raw)

        if q and only_matches and (q not in text.lower()):
            continue

        out.append(f"### {i}. `{role}`")
        calls = extract_tool_calls(m)
        if calls:
            out.append("**tool_calls:**")
            out.append("```json")
            out.append(json.dumps(calls, indent=2, ensure_ascii=False, default=str))
            out.append("```")

        if text.strip():
            out.append("```")
            out.append(text[:8000] + ("\n...[truncated]..." if len(text) > 8000 else ""))
            out.append("```")
        else:
            out.append("_(empty content — tool-call-only or structured output)_")

        # ToolMessage extras
        if role == "tool":
            tcid = getattr(m, "tool_call_id", None)
            if tcid:
                out.append(f"- tool_call_id: `{tcid}`")

    return "\n".join(out)


def render_prompt_tab(state: Dict[str, Any], search: str = "", only_matches: bool = False) -> str:
    pm = get_prompt_messages_from_state(state)
    if pm:
        return render_messages(pm, "Prompt Messages", search, only_matches)

    # Fallback to persisted prompt artifacts
    fallback = get_prompt_text_fallback(state)
    if fallback.strip():
        return "## Prompt (artifact fallback)\n```text\n" + fallback[:12000] + ("\n...[truncated]..." if len(fallback) > 12000 else "") + "\n```"

    return "_(Prompt not available: ensure context node sets prompt_messages OR add persist_prompt_artifact_node)_"

In [ ]:
def render_history(hist) -> str:
    if not hist:
        return "_(history empty)_"

    out = []
    for i, m in enumerate(hist):
        role = _role(m)
        raw = _content(m)
        content = normalize_content(raw)


        out.append(f"### {i}. `{role}`  (≈{msg_tokens(m)} toks)")

        # 1) show tool calls if present (important for AIMessage)
        tool_calls = getattr(m, "tool_calls", None) or []
        if tool_calls:
            out.append("**tool_calls:**")
            out.append("```json")
            out.append(_pretty(tool_calls))
            out.append("```")

        # 2) show content if non-empty
        if content.strip():
            out.append("```")
            out.append(_truncate(content, 6000))
            out.append("```")
        else:
            # 3) for empty AI content, show a placeholder so it doesn't look broken
            out.append("_(empty content — likely a tool-call-only AIMessage)_")

        # 4) if it's a ToolMessage, show tool_call_id
        if m.__class__.__name__ == "ToolMessage":
            tcid = getattr(m, "tool_call_id", None)
            if tcid:
                out.append(f"- tool_call_id: `{tcid}`")

    return "\n".join(out)

def render_prompt(pm) -> str:
    if not pm:
        return "_(messages empty)_"

    out = []
    total = 0
    for i, m in enumerate(pm):
        t = msg_tokens(m)
        total += t

        role = _role(m)
        content = _content(m)
        out.append(f"### {i}. `{role}`  (≈{t} toks)")

        tool_calls = getattr(m, "tool_calls", None) or []
        if tool_calls:
            out.append("**tool_calls:**")
            out.append("```json")
            out.append(_pretty(tool_calls))
            out.append("```")

        if content.strip():
            out.append("```")
            out.append(_truncate(content, 6000))
            out.append("```")
        else:
            out.append("_(empty content)_")

    out.insert(0, f"**Estimated prompt tokens:** ≈{total}")
    return "\n".join(out)
def render_telemetry(tel) -> str:
    if not tel:
        return "_(telemetry empty — wrap nodes with instrumentor to populate)_"
    return "```json\n" + _pretty_json(tel[-50:]) + "\n```"

def render_runtime(rt) -> str:
    if not rt:
        return "_(runtime empty)_"
    return "```json\n" + _pretty_json(rt) + "\n```"

def render_memory(mem) -> str:
    if not mem:
        return "_(memory empty)_"
    return "```json\n" + _pretty_json(mem) + "\n```"

def render_diff(d) -> str:
    return "```json\n" + _pretty_json(d) + "\n```"

def render_tool_inspector(hist) -> str:
    if not hist:
        return "_(no history)_"
    # find last AI message with tool_calls
    for m in reversed(hist):
        if m.__class__.__name__ == "AIMessage":
            calls = extract_tool_calls(m)
            if calls:
                return "### Last tool calls\n```json\n" + _pretty_json(calls) + "\n```"
    return "_(no tool calls found in AI messages)_"

def prompt_text(pm) -> str:
    if not pm:
        return ""
    lines = []
    for m in pm:
        lines.append(f"{_role(m)}:\n{_content(m)}\n")
    return "\n".join(lines)

def render_prompt_diff(prev_pm, cur_pm) -> str:
    a = prompt_text(prev_pm).splitlines()
    b = prompt_text(cur_pm).splitlines()
    diff = difflib.unified_diff(a, b, fromfile="prev_prompt", tofile="cur_prompt", lineterm="")
    txt = "\n".join(diff)
    if not txt.strip():
        return "_(no prompt diff)_"
    return "```diff\n" + _truncate(txt, 12000) + "\n```"

In [ ]:
def render_rich_step(cur: dict, prev: dict) -> str:
    """
    Returns ANSI string containing rich-rendered view of current step.
    """
    console = Console(record=True, width=110)  # record=True lets us export text
    rt = safe_get(cur, "runtime", {}) or {}
    tel = safe_get(cur, "telemetry", []) or []
    hist = get_history_from_state(cur)

    # --- Header panel ---
    title = f"Step {safe_get(cur,'_step',None) or ''}".strip()
    hdr = Table.grid(expand=True)
    hdr.add_column(justify="left")
    hdr.add_column(justify="right")
    hdr.add_row(
        f"🧠 run_id={rt.get('run_id','?')}  turn={rt.get('turn_index',0)}  after_tool={rt.get('after_tool',False)}",
        f"history={len(hist)}  telemetry={len(tel)}"
    )
    console.print(Panel(hdr, title="📍 Snapshot", border_style="cyan"))

    # --- Interrupts if any ---
    intr = cur.get("__interrupt__")
    if intr:
        console.print(Panel(str(intr)[:2000], title="⏸️ Interrupt", border_style="yellow"))

    # --- Recent history tree (last N) ---
    tree = Tree("🧾 Recent messages")
    for m in hist[-8:]:
        role = m.__class__.__name__.replace("Message", "").lower()
        icon = {"human":"👤","ai":"🤖","tool":"🧰","system":"⚙️"}.get(role, "💬")
        content = normalize_content(getattr(m, "content", None))
        content = content.strip() if isinstance(content, str) else str(content)
        if len(content) > 240:
            content = content[:240] + "…"
        # show tool calls summary if AI
        extra = ""
        if m.__class__.__name__ == "AIMessage":
            tc = extract_tool_calls(m)
            if tc:
                extra = f"  🔧 tool_calls={len(tc)}"
        tree.add(f"{icon} {role}{extra} :: {content}")
    console.print(tree)

    # --- Telemetry last entry (node timing/status) ---
    if tel:
        last = tel[-1]
        t = Table(title="📊 Last telemetry entry", box=box.SIMPLE, show_lines=False)
        t.add_column("key", style="bold")
        t.add_column("value")
        for k in ["node","status","elapsed_ms","turn_index","run_id"]:
            if k in last:
                t.add_row(k, str(last.get(k)))
        # error
        err = last.get("error") or last.get("error", "")
        if isinstance(err, dict):
            t.add_row("error.type", str(err.get("type","")))
            t.add_row("error.class", str(err.get("class","")))
            t.add_row("error.msg", str(err.get("message",""))[:300])
        elif err:
            t.add_row("error", str(err)[:300])
        console.print(t)

    return console.export_text(clear=False)  # ANSI-ish plain text (safe)

In [ ]:
class SotaGraphUI:
    def __init__(self, steps: List[Step]):
        self.steps = steps
        self.ptr = 0

        # Outputs
        self.out_history = widgets.Output()
        self.out_prompt = widgets.Output()
        self.out_diff = widgets.Output()
        self.out_updates = widgets.Output()
        self.out_tool = widgets.Output()
        self.out_runtime = widgets.Output()
        self.out_memory = widgets.Output()
        self.out_telemetry = widgets.Output()
        self.out_rich = widgets.Output()

        # Controls
        self.btn_next = widgets.Button(description="Next", button_style="primary")
        self.btn_prev = widgets.Button(description="Prev")
        self.btn_play = widgets.ToggleButton(description="Play ▶", value=False, button_style="")
        self.btn_clear = widgets.Button(description="Clear Tab")
        self.btn_reset = widgets.Button(description="Reset ⟲", button_style="warning")

        self.speed = widgets.IntSlider(value=300, min=50, max=2000, step=50, description="ms/step", continuous_update=False)
        self.slider = widgets.IntSlider(value=0, min=0, max=max(0, len(steps)-1), step=1, description="step", continuous_update=False)

        self.search = widgets.Text(value="", description="search", placeholder="find in prompt/history")
        self.chk_only_matches = widgets.Checkbox(value=False, description="only matching msgs")

        self.tabs = widgets.Tab(children=[
            widgets.VBox([self.out_history]),
            widgets.VBox([self.out_prompt]),
            widgets.VBox([self.out_diff]),
            widgets.VBox([self.out_tool]),
            widgets.VBox([self.out_updates]),
            widgets.VBox([self.out_runtime]),
            widgets.VBox([self.out_memory]),
            widgets.VBox([self.out_telemetry]),
            widgets.VBox([self.out_rich]),          # NEW
        ])
        titles = ["History", "Prompt", "Prompt Diff", "Tools", "Diff/Updates", "Runtime", "Memory", "Telemetry", "Rich"]
        for i, t in enumerate(titles):
            self.tabs.set_title(i, t)

        # Header
        self.hdr = widgets.HTML()

        # Wire events
        self.btn_next.on_click(lambda _: self.step_to(self.ptr + 1))
        self.btn_prev.on_click(lambda _: self.step_to(self.ptr - 1))
        self.btn_reset.on_click(lambda _: self.step_to(0))
        self.btn_clear.on_click(lambda _: self._clear_active_tab())
        self.slider.observe(lambda ch: self.step_to(ch["new"]), names="value")
        self.search.observe(lambda ch: self.render(), names="value")
        self.chk_only_matches.observe(lambda ch: self.render(), names="value")
        self.btn_play.observe(lambda ch: self._on_play(ch["new"]), names="value")

        self.ui = widgets.VBox([
            widgets.HBox([self.btn_prev, self.btn_next, self.btn_play, self.btn_clear, self.btn_reset]),
            widgets.HBox([self.slider, self.speed]),
            widgets.HBox([self.search, self.chk_only_matches, self.hdr]),
            self.tabs
        ])

        self.render()

    def show(self):
        display(self.ui)

    def _clear_active_tab(self):
        # If autoplay is running, pause it so the cleared tab stays cleared
        self.btn_play.value = False

        outs = [
            self.out_history, self.out_prompt, self.out_diff, self.out_tool,
            self.out_updates, self.out_runtime, self.out_memory, self.out_telemetry,
            self.out_rich,  # NEW
        ]
        out = outs[self.tabs.selected_index]

        # Correct way: clear the Output widget buffer
        out.clear_output(wait=True)

    def step_to(self, idx: int):
        idx = max(0, min(idx, len(self.steps)-1))
        self.ptr = idx
        self.slider.value = idx
        self.render()

    def _on_play(self, on: bool):
        if not on:
            return
        # simple play loop (runs in notebook cell execution context)
        while self.btn_play.value and self.ptr < len(self.steps)-1:
            self.step_to(self.ptr + 1)
            time.sleep(self.speed.value / 1000.0)
        self.btn_play.value = False

    def _filter_msgs(self, msgs: List[Any]) -> List[Any]:
        q = (self.search.value or "").strip().lower()
        if not q:
            return msgs
        if not self.chk_only_matches.value:
            return msgs
        out = []
        for m in msgs:
            if q in (_content(m).lower()):
                out.append(m)
        return out

    def render(self):
        if not self.steps:
            self.hdr.value = "<b>No steps recorded</b>"
            return
        cur = self.steps[self.ptr].state  # full state dict (or snapshot dict)
        prev = self.steps[self.ptr-1].state if self.ptr > 0 else {}
        hist = get_history_from_state(cur)
        prompt_md = render_prompt_tab(cur, self.search.value, self.chk_only_matches.value)
        pm = safe_get(cur, "messages", [])
        pm_prev = safe_get(prev, "messages", [])
        tel = safe_get(cur, "telemetry", [])
        rt = safe_get(cur, "runtime", {})
        mem = safe_get(cur, "memory", {})
        diff = self.steps[self.ptr].diff

        # Header stats
        prompt_tok = sum(msg_tokens(m) for m in (pm or []))
        hist_len = len(safe_get(cur, "history", []))
        self.hdr.value = f"<b>step</b> {self.ptr}/{len(self.steps)-1} | <b>history</b> {hist_len} msgs | <b>prompt</b> ≈{prompt_tok} toks"


        with self.out_history:
            clear_output(wait=True)
            display(Markdown(render_messages(hist, "History", self.search.value, self.chk_only_matches.value)))

        with self.out_prompt:
            clear_output(wait=True)
            display(Markdown(prompt_md))

        with self.out_diff:
            clear_output(wait=True)
            display(Markdown(render_prompt_diff(pm_prev, pm)))

        with self.out_tool:
            clear_output(wait=True)
            display(Markdown(render_tool_inspector(safe_get(cur, "history", []))))

        with self.out_updates:
            clear_output(wait=True)
            display(Markdown(render_diff(diff)))

        with self.out_runtime:
            clear_output(wait=True)
            display(Markdown(render_runtime(rt)))

        with self.out_memory:
            clear_output(wait=True)
            display(Markdown(render_memory(mem)))

        with self.out_telemetry:
            clear_output(wait=True)
            display(Markdown(render_telemetry(tel)))

        with self.out_rich:
            clear_output(wait=True)
            pretty = render_rich_step(cur, prev)
            # Render as preformatted text
            display(widgets.HTML(f"<pre style='font-size:12px; line-height:1.25'>{pretty}</pre>"))

In [ ]:
ResumeAnswer = Union[str, int, float, bool, Dict[str, Any], List[Any]]

def _extract_interrupt_payload(state: Dict[str, Any]) -> Optional[Any]:
    """
    LangGraph typically stores interrupts under '__interrupt__' in the returned state.
    This helper normalizes it into a payload you can render.
    """
    intr = state.get("__interrupt__")
    if not intr:
        return None

    # Some versions store a list of objects like {"value": {...}, ...}
    if isinstance(intr, list) and len(intr) > 0:
        first = intr[0]
        if isinstance(first, dict) and "value" in first:
            return first["value"]
        return first
    return intr

def record_run(
    app,
    initial_state: Dict[str, Any],
    keys: Optional[List[str]] = None,
    config: Optional[Dict[str, Any]] = None,
    # If provided, called when an interrupt is hit. Must return the resume answer.
    on_interrupt: Optional[Callable[[Any, Dict[str, Any]], ResumeAnswer]] = None,
    # If True, auto-resume using on_interrupt; if False, stop at first interrupt.
    auto_resume: bool = True,
    # Safety: prevent infinite pause/resume loops
    max_interrupts: int = 10,
) -> List[Step]:
    """
    Reliable recorder for LangGraph with HITL interrupts.

    - Uses stream_mode='values' to capture full state snapshots.
    - Detects interrupts via state['__interrupt__'].
    - If an interrupt occurs:
        - if auto_resume and on_interrupt provided: resumes via Command(resume=answer)
        - else: records the interrupt and returns (caller/UI resumes later)

    IMPORTANT:
    - For resume to work, app must be compiled with a checkpointer.
    - config must include a stable thread_id:
        config={"configurable":{"thread_id":"..."}}
    """
    if keys is None:
        keys = ["history","messages","messages","input_messages","llm_input","telemetry","runtime","memory","__interrupt__"]

    # Ensure we have a thread_id for checkpointed resume
    if config is None:
        config = {"configurable": {"thread_id": f"thread-{uuid.uuid4()}"}}
    else:
        config = dict(config)
        cfg = dict(config.get("configurable", {}) or {})
        cfg.setdefault("thread_id", f"thread-{uuid.uuid4()}")
        config["configurable"] = cfg

    steps: List[Step] = []
    prev_snap: Dict[str, Any] = {}
    idx = 0
    interrupts_seen = 0

    def _consume_stream(input_obj):
        nonlocal idx, prev_snap, interrupts_seen
        for full_state in app.stream(input_obj, config=config, stream_mode="values"):
            snap = _shallow_snapshot(full_state, keys)
            d = _diff_states(prev_snap, snap) if idx > 0 else {"note": "initial snapshot"}
            steps.append(Step(idx=idx, state=snap, diff=d))
            prev_snap = snap
            idx += 1

            payload = _extract_interrupt_payload(full_state)
            if payload is not None:
                interrupts_seen += 1
                if interrupts_seen > max_interrupts:
                    # record a marker and stop
                    steps.append(Step(
                        idx=idx,
                        state={"__interrupt__": payload, "note": "max_interrupts exceeded"},
                        diff={"note": "stopped"}
                    ))
                    return payload, False  # interrupted, not resumed

                if not (auto_resume and on_interrupt):
                    return payload, False  # interrupted, not resumed

                # Ask UI/caller for answer
                answer = on_interrupt(payload, full_state)
                # Resume immediately
                return payload, answer

        return None, None  # completed without interrupt

    # 1) Run from initial_state
    payload, resume = _consume_stream(initial_state)

    # 2) If interrupted and we have an answer, resume loop
    while payload is not None and resume is not False:
        # If resume is True-ish but not an answer, that’s a bug; enforce actual answer value
        answer = resume
        payload, resume = _consume_stream(Command(resume=answer))

    return steps

In [ ]:
def on_interrupt(payload, full_state):
    # payload includes your question/context/choices
    print("INTERRUPT:", payload)
    return "yes"  # or return selected option, dict, etc.


In [ ]:
state = {'history': [HumanMessage('Randomly decide on a ML application to develop a model for and then develop the model.')]}

In [ ]:
app = build_app(llm_with_tools, prompt_lib,
                budget_policy=budget_policy,
                tool_log_policy=tool_policy,
                summary_policy = summary_policy,
                tools = tools)

In [ ]:
steps = record_run(
    app,
    state,
    config={"configurable": {"thread_id": "t1"}},
    on_interrupt=on_interrupt,
    auto_resume=True
)

INTERRUPT: (Interrupt(value={'type': 'confirm', 'reason': 'plan_created', 'question': "I have chosen 'Iris species classification' randomly. Shall I proceed with the plan?", 'choices': None, 'context': "The chosen ML application is 'Iris species classification'.\nPlan:\n1. Load the Iris dataset.\n2. Preprocess the data.\n3. Train a Random Forest model.\n4. Evaluate and summarize results.", 'default': None}, id='ffb096ba982f942ac5c816c4a5361abf'),)


In [ ]:
ui = SotaGraphUI(steps)
ui.show()

## Agent Console UI

In [ ]:
ResumeAnswer = Union[str, int, float, bool, Dict[str, Any], List[Any]]

def _extract_interrupt_payload(state: Dict[str, Any]) -> Optional[Any]:
    intr = state.get("__interrupt__")
    if not intr:
        return None
    if isinstance(intr, list) and intr:
        x = intr[0]
        if isinstance(x, dict) and "value" in x:
            return x["value"]
        return x
    return intr

def _as_payload(payload: Any) -> Dict[str, Any]:
    if isinstance(payload, dict):
        return {
            "type": payload.get("type", "clarify"),
            "reason": payload.get("reason", ""),
            "question": payload.get("question", ""),
            "choices": payload.get("choices") or [],
            "context": payload.get("context", ""),
            "default": payload.get("default", ""),
        }
    return {"type": "clarify", "reason": "", "question": str(payload), "choices": [], "context": "", "default": ""}

def _last_ai_text_from_state(state: Dict[str, Any]) -> Optional[str]:
    hist = state.get("history", []) or []
    for m in reversed(hist):
        if m.__class__.__name__ == "AIMessage":
            return normalize_model_text(getattr(m, "content", None))
    return None

def _md_escape(s: str) -> str:
    return s or ""

def _pill(text: str, bg="#eee", fg="#111") -> str:
    return f"<span style='display:inline-block;padding:2px 8px;border-radius:999px;background:{bg};color:{fg};font-size:12px;margin-right:6px'>{html.escape(text)}</span>"

def _card_html(title: str, body_html: str, border="#ddd") -> str:
    return f"""
    <div style="border:1px solid {border}; border-radius:14px; padding:12px 12px; background:#fff; box-shadow:0 1px 2px rgba(0,0,0,0.06);">
      <div style="font-weight:700; font-size:14px; margin-bottom:8px;">{html.escape(title)}</div>
      {body_html}
    </div>
    """

def normalize_interrupt(raw: Any) -> Optional[Dict[str, Any]]:
    """
    Accepts anything LangGraph might stick into __interrupt__ (tuple/list/Interrupt/dict)
    and returns a clean dict:
      {"type","reason","question","choices","context","default","id"}
    """
    if not raw:
        return None

    # __interrupt__ often is list/tuple of Interrupt objects
    if isinstance(raw, (list, tuple)) and raw:
        raw = raw[0]

    # Interrupt object has .value and .id
    value = None
    intr_id = None
    if hasattr(raw, "value"):
        value = getattr(raw, "value", None)
        intr_id = getattr(raw, "id", None)
    elif isinstance(raw, dict) and "value" in raw:
        value = raw.get("value")
        intr_id = raw.get("id")
    else:
        value = raw

    if not isinstance(value, dict):
        return {
            "type": "clarify",
            "reason": "",
            "question": str(value),
            "choices": [],
            "context": "",
            "default": "",
            "id": intr_id,
        }

    return {
        "type": value.get("type", "clarify"),
        "reason": value.get("reason", ""),
        "question": value.get("question", ""),
        "choices": value.get("choices") or [],
        "context": value.get("context", ""),
        "default": value.get("default", ""),
        "id": intr_id,
    }

In [ ]:
def render_mdish_html(text: str) -> str:
    """
    Minimal markdown-ish rendering:
      - ``` code fences -> styled <pre>
      - blank lines -> spacing
      - keeps normal text readable
    """
    text = text or ""
    parts = re.split(r"```", text)
    out = []
    for i, chunk in enumerate(parts):
        if i % 2 == 1:
            lines = chunk.splitlines()
            # optional language tag
            if lines and re.match(r"^[a-zA-Z0-9_+-]+$", lines[0].strip()):
                lines = lines[1:]
            code = "\n".join(lines)
            out.append(
                "<pre style='background:#0f172a;color:#e2e8f0;padding:10px;border-radius:12px;"
                "overflow:auto;white-space:pre; font-size:12px; line-height:1.35'>"
                f"{html.escape(code)}"
                "</pre>"
            )
        else:
            safe = html.escape(chunk)
            safe = safe.replace("\n\n", "<br><br>").replace("\n", "<br>")
            out.append(f"<div style='font-size:14px; line-height:1.45'>{safe}</div>")
    return "".join(out)
def _render_answer_html(text: str) -> str:
    """
    Minimal markdown-ish renderer to HTML for chat bubble.
    Supports:
      - ```code``` blocks
      - bullet lists
      - paragraphs
    """
    text = text or ""
    parts = re.split(r"```", text)
    out = []
    for i, chunk in enumerate(parts):
        if i % 2 == 1:
            # code block (may start with language)
            lines = chunk.splitlines()
            if lines and re.match(r"^[a-zA-Z0-9_+-]+$", lines[0].strip()):
                lines = lines[1:]
            code = "\n".join(lines)
            out.append(f"<pre style='background:#0f172a;color:#e2e8f0;padding:10px;border-radius:12px;overflow:auto;white-space:pre;'>{html.escape(code)}</pre>")
        else:
            # normal text: simple bullets + paragraphs
            lines = chunk.splitlines()
            buf = []
            for ln in lines:
                ln = ln.rstrip()
                if ln.strip().startswith(("- ", "* ")):
                    buf.append(f"• {ln.strip()[2:]}")
                else:
                    buf.append(ln)
            para = "<br/>".join(html.escape(x) for x in buf if x is not None)
            if para.strip():
                out.append(f"<div style='white-space:pre-wrap; font-size:14px; line-height:1.4'>{para}</div>")
    return "".join(out)

In [ ]:
def normalize_model_text(content: Any) -> str:
    """
    Handles:
      - plain string
      - list of parts [{'type':'text','text':...}, ...]
      - dict with 'text'
      - anything else -> str(...)
    """
    if content is None:
        return ""

    # Most common
    if isinstance(content, str):
        return content

    # Gemini / structured parts
    if isinstance(content, list):
        texts = []
        for p in content:
            if isinstance(p, dict):
                if p.get("type") == "text" and "text" in p:
                    texts.append(p["text"])
                elif "text" in p:
                    texts.append(str(p["text"]))
            else:
                # fallback for unexpected part types
                texts.append(str(p))
        return "\n".join([t for t in texts if t])

    # Some wrappers use dict
    if isinstance(content, dict):
        if "text" in content:
            return str(content["text"])
        # fallback: join common keys if present
        for k in ("content", "message"):
            if k in content:
                return normalize_model_text(content[k])
        return str(content)

    return str(content)

In [ ]:
def interrupt_card_html(p: Dict[str, Any]) -> str:
    import html
    def pill(t, bg, fg):
        return f"<span style='padding:2px 8px;border-radius:999px;background:{bg};color:{fg};font-size:12px;margin-right:6px'>{html.escape(t)}</span>"

    pills = pill(p["type"], "#E8F0FE", "#174EA6")
    if p.get("reason"):
        pills += pill(p["reason"], "#FCE8E6", "#A50E0E")

    ctx = p.get("context","") or ""
    ctx_html = ""
    if ctx.strip():
        ctx_html = f"""
        <details style="margin-top:10px;">
          <summary style="cursor:pointer; opacity:0.85;">Context</summary>
          <pre style="white-space:pre-wrap; background:#f7f7f7; padding:10px; border-radius:10px; max-height:260px; overflow:auto;">{html.escape(ctx)}</pre>
        </details>
        """

    return f"""
      <div style="margin-bottom:8px;">{pills}</div>
      <div style="font-size:14px; line-height:1.35;">{html.escape(p.get("question",""))}</div>
      {ctx_html}
    """

In [ ]:
class ChatGPTColabUI:
    """
    Minimal, user-facing ChatGPT-style UI for a LangGraph agent with HITL via interrupts.

    Requirements:
    - app is compiled with a checkpointer
    - you run with config={"configurable":{"thread_id":...}}
    - interrupts appear under state["__interrupt__"]
    """

    def __init__(self, app, make_initial_state, title="Chat UI"):
        self.app = app
        self.make_initial_state = make_initial_state
        self.thread_id = f"thread-{uuid.uuid4()}"

        # --- UI elements ---
        self.out_chat = widgets.Output(layout=widgets.Layout(border="0px solid #ddd", padding="6px"))
        self.txt = widgets.Textarea(
            placeholder="Message…",
            layout=widgets.Layout(width="100%", height="80px"),
        )
        self.btn_send = widgets.Button(description="Send", button_style="primary")
        self.btn_reset = widgets.Button(description="New chat", button_style="warning")
        self.btn_copy_last = widgets.Button(description="Copy last", icon="copy")
        self.btn_regen = widgets.Button(description="Regenerate", icon="refresh")

        # HITL widgets (shown inline when needed)
        self.hitl_box = widgets.VBox()
        self.hitl_input = widgets.Textarea(layout=widgets.Layout(width="100%", height="70px"))
        self.hitl_dropdown = widgets.Dropdown(options=[], value=None, layout=widgets.Layout(width="100%"))
        self.hitl_multi = widgets.SelectMultiple(options=[], value=(), layout=widgets.Layout(width="100%", height="110px"))
        self.btn_approve = widgets.Button(description="Approve", button_style="success")
        self.btn_reject = widgets.Button(description="Reject", button_style="danger")
        self.btn_submit = widgets.Button(description="Submit", button_style="primary")
        self.btn_submit_edits = widgets.Button(description="Submit edits", button_style="")

        self.status = widgets.HTML(value="")

        self.ui = widgets.VBox([
            widgets.HBox([widgets.HTML(f"<h3 style='margin:6px 0'>{title}</h3>"), self.status],
                        layout=widgets.Layout(justify_content="space-between")),
            self.out_chat,
            self.hitl_box,
            widgets.HBox([
                self.txt,
                widgets.VBox([self.btn_send, self.btn_regen, self.btn_copy_last, self.btn_reset],
                            layout=widgets.Layout(width="140px"))
            ]),
        ])
        self._md_bubble_out = widgets.Output()

        # Wire
        self.btn_send.on_click(lambda _: self._on_send())
        self.btn_reset.on_click(lambda _: self._reset())
        self.btn_approve.on_click(lambda _: self._resume("yes"))
        self.btn_reject.on_click(lambda _: self._resume("no"))
        self.btn_submit.on_click(lambda _: self._resume(self._hitl_value()))
        self.btn_submit_edits.on_click(lambda _: self._resume(self._hitl_value()))
        self.btn_copy_last.on_click(lambda _: self._copy_last())
        self.btn_regen.on_click(lambda _: self._regenerate())

        self._chat_system("Ready. Ask me anything.")

    def show(self):
        display(self.ui)

    # ---------- Chat rendering ----------
    def _bubble(self, who: str, text: str, *, is_interrupt: bool = False):
        text = text or ""
        body = render_mdish_html(text) if who != "You" else (
            f"<div style='white-space:pre-wrap; font-size:14px; line-height:1.45'>{html.escape(text)}</div>"
        )

        if who == "You":
            bg = "#DCF8C6"; align = "flex-end"; border = "#cdebb5"
        else:
            bg = "#FFFFFF"; align = "flex-start"; border = "#e6e6e6"

        with self.out_chat:
            display(widgets.HTML(f"""
            <div style="display:flex; justify-content:{align}; margin:8px 0;">
            <div style="max-width:86%; background:{bg}; padding:12px 12px;
                        border-radius:14px; border:1px solid {border};">
                <div style="font-size:12px; opacity:0.55; margin-bottom:6px;">
                {html.escape(who)}
                </div>
                {body}
            </div>
            </div>
            """))
    def _chat_system(self, text: str):
        self._bubble("Agent", text)

    def _set_status(self, text: str):
        self.status.value = f"<span style='opacity:0.7; font-size:12px'>{text}</span>"

    # ---------- Running the graph ----------
    def _config(self):
        return {"configurable": {"thread_id": self.thread_id}}

    def _run(self, input_obj):
        """
        Run until completion or interrupt.
        Returns the last state snapshot.
        """
        last_state = None
        for st in self.app.stream(input_obj, config=self._config(), stream_mode="values"):
            last_state = st
            # if you store runtime flags like after_tool / last_tool, show them:
            rt = st.get("runtime", {}) or {}
            if rt.get("after_tool"):
                self._set_status("Working…")
        return last_state or {}

    def _on_send(self):
        text = (self.txt.value or "").strip()
        if not text:
            return
        self.txt.value = ""
        self._bubble("You", text)

        self._set_status("Thinking…")
        init_state = self.make_initial_state(text)

        last = self._run(init_state)
        self._handle_result(last)
        self._set_status("")
        self._last_user_text = text

    def _handle_result(self, last_state: Dict[str, Any]):
        payload = normalize_interrupt(last_state.get("__interrupt__"))
        if payload:
            self._show_hitl(payload)
            return

        reply = _last_ai_text_from_state(last_state)
        if reply:
            self._bubble("Agent", reply)
        else:
            self._bubble("Agent", "(no text output)")
        self._last_agent_text = reply
        self._hide_hitl()

    def _copy_last(self):
        txt = getattr(self, "_last_agent_text", "") or ""
        if not txt:
            return
        # Colab-friendly: copy by putting into clipboard via JS
        from IPython.display import Javascript
        safe = txt.replace("\\", "\\\\").replace("`","\\`").replace("${","\\${")
        display(Javascript(f"""
        navigator.clipboard.writeText(`{safe}`);
        """))
        self._set_status("Copied.")

    def _regenerate(self):
        # Regenerate = re-run last user message.
        # You need to store it.
        last_user = getattr(self, "_last_user_text", None)
        if not last_user:
            return
        self._bubble("You", "(regenerate)")
        init_state = self.make_initial_state(last_user)
        self._set_status("Thinking…")
        last = self._run(init_state)
        self._handle_result(last)
        self._set_status("")

    # ---------- HITL ----------
    def _show_hitl(self, payload: Any):
        p = payload if isinstance(payload, dict) else _as_payload(payload)
        kind = p["type"]
        question = p["question"] or "(no question provided)"
        reason = (p.get("reason") or "").strip()
        ctx = p.get("context") or ""
        choices = p.get("choices") or []

        # Reason + kind pills
        pills = _pill(f"{kind}", bg="#E8F0FE", fg="#174EA6")
        if reason:
            pills += _pill(reason, bg="#FCE8E6", fg="#A50E0E")

        # Collapsible context (HTML details)
        ctx_html = ""
        if ctx.strip():
            safe_ctx = html.escape(ctx[:6000])
            ctx_html = f"""
            <details style="margin-top:10px;">
            <summary style="cursor:pointer; opacity:0.85;">Context</summary>
            <pre style="white-space:pre-wrap; background:#f7f7f7; padding:10px; border-radius:10px; overflow:auto; max-height:260px;">{safe_ctx}</pre>
            </details>
            """

        body = f"""
        <div style="margin-bottom:8px;">{pills}</div>
        <div style="font-size:14px; line-height:1.35; white-space:pre-wrap;">{html.escape(question)}</div>
        {ctx_html}
        """

        # Render interrupt card as a bubble
        with self.out_chat:
            display(widgets.HTML(f"""
            <div style="display:flex; justify-content:flex-start; margin:8px 0;">
            <div style="max-width:86%;">
                {_card_html("⏸ Needs your input", body, border="#f3c969")}
            </div>
            </div>
            """))

        # Build input UI under chat (kept separate, but visually clean)
        ui_children = []
        if kind == "confirm":
            self.hitl_input.value = ""
            ui_children = [
                widgets.HBox([self.btn_approve, self.btn_reject]),
                widgets.HTML("<div style='opacity:0.7;font-size:12px;margin-top:6px;'>Or type edits/notes and submit:</div>"),
                self.hitl_input,
                self.btn_submit_edits,
            ]
        elif kind == "pick_one":
            self.hitl_dropdown.options = choices
            self.hitl_dropdown.value = choices[0] if choices else None
            ui_children = [self.hitl_dropdown, self.btn_submit]
        elif kind == "pick_many":
            self.hitl_multi.options = choices
            self.hitl_multi.value = ()
            ui_children = [self.hitl_multi, self.btn_submit]
        else:  # clarify/free text
            self.hitl_input.value = p.get("default", "") or ""
            ui_children = [self.hitl_input, self.btn_submit]

        self.hitl_box.children = tuple(ui_children)

    def _hide_hitl(self):
        self.hitl_box.children = tuple()

    def _hitl_value(self) -> ResumeAnswer:
        # determine current HITL input value based on visible widget
        if self.hitl_box.children:
            for ch in self.hitl_box.children:
                if ch is self.hitl_dropdown:
                    return self.hitl_dropdown.value
                if ch is self.hitl_multi:
                    return list(self.hitl_multi.value)
        return (self.hitl_input.value or "").strip()

    def _resume(self, answer: ResumeAnswer):
        if answer is None or (isinstance(answer, str) and not answer.strip()):
            return
        self._bubble("You", f"(approval) {answer}")
        self._hide_hitl()

        self._set_status("Continuing…")
        last = self._run(Command(resume=answer))
        self._handle_result(last)
        self._set_status("")

    # ---------- Reset ----------
    def _reset(self):
        self.thread_id = f"thread-{uuid.uuid4()}"
        self._hide_hitl()
        self.out_chat.clear_output(wait=True)
        self._chat_system("New chat started.")

In [ ]:
def make_initial_state(user_text: str) -> Dict[str, Any]:
    return {
        "history": [HumanMessage(content=user_text)],
        "memory": {},
        "runtime": {"run_id": "colab", "turn_index": 0},
        "skills": [],  # or discover_skills(...)
    }

chat_ui = ChatGPTColabUI(app, make_initial_state, title="My Agent")
chat_ui.show()

FileNotFoundError: Skill not found: list_skills